# Entrega 2: indexação das descrições comerciais

Este notebook, sem arquivo Python auxiliar, indexa as 246 descrições de `dados/entradas/v1`, uma por arquivo GoBR. O código original fica ligado ao mesmo ID e é devolvido na busca. No Colab, os dados são clonados automaticamente do GitHub quando necessário; não é necessário carregar o Qwen.

Os embeddings recebem somente a descrição em português. A consulta do usuário deve ser escrita no mesmo tipo de linguagem. O diagnóstico interrompe a execução se uma descrição estiver vazia ou sem fonte.

In [1]:
%pip install -q "llama-index-core>=0.13,<0.15" "llama-index-embeddings-huggingface>=0.7,<0.8" "sentence-transformers>=3,<6" "transformers>=4.41,<5"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## Localizar o repositório

No Colab, esta célula baixa automaticamente os dados do repositório público no GitHub. Se você já tiver um checkout, ele será reutilizado. Opcionalmente, defina `GOBR_REPO_DIR` para usar uma pasta específica.

In [2]:
from pathlib import Path
import os
import subprocess

URL_REPOSITORIO = 'https://github.com/buenofgustavo/projeto-sistemas-multiagentes.git'
NOME_REPOSITORIO = 'projeto-sistemas-multiagentes'
def contem_dados(pasta):
    return (pasta / 'dados/entradas/v1').is_dir() and (pasta / 'dados/referencias').is_dir()

pasta_definida = os.environ.get('GOBR_REPO_DIR')
if pasta_definida:
    REPO_DIR = Path(pasta_definida).expanduser().resolve()
    if not contem_dados(REPO_DIR):
        raise FileNotFoundError(f'GOBR_REPO_DIR não contém os dados esperados: {REPO_DIR}')
else:
    candidatos = [Path.cwd(), Path.cwd() / NOME_REPOSITORIO, Path('/workspace') / NOME_REPOSITORIO]
    REPO_DIR = next((p.resolve() for p in candidatos if contem_dados(p)), None)
    if REPO_DIR is None:
        destino = Path.cwd() / NOME_REPOSITORIO
        if destino.exists():
            destino = next((Path.cwd() / f'{NOME_REPOSITORIO}_{n}' for n in range(2, 100)
                            if not (Path.cwd() / f'{NOME_REPOSITORIO}_{n}').exists()), None)
            if destino is None:
                raise RuntimeError('Não há pasta livre para clonar o repositório')
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', URL_REPOSITORIO, str(destino)], check=True)
        REPO_DIR = destino.resolve()
        if not contem_dados(REPO_DIR):
            raise RuntimeError(f'O repositório clonado não contém as pastas esperadas: {REPO_DIR}')
print('Repositório:', REPO_DIR)


Repositório: /content/projeto-sistemas-multiagentes


In [3]:
from __future__ import annotations

import hashlib
import json
from dataclasses import dataclass
from importlib.metadata import version
from pathlib import Path


MODELO_EMBEDDING = "BAAI/bge-m3"
VERSAO_PIPELINE = "descricoes-comerciais-v1-246"
LIMITE_TOKENS = 8192


@dataclass(frozen=True)
class Exemplo:
    id: str
    explicacao: str
    codigo: str
    arquivo: str


def carregar_exemplos(repo_dir: Path) -> tuple[list[Exemplo], list[str]]:
    """Liga cada descrição comercial ao GoBR pelo nome, sem embutir código."""
    pasta = repo_dir / "dados" / "referencias"
    pasta_descricoes = repo_dir / "dados" / "entradas" / "v1"
    arquivos = {p.stem: p for p in pasta.glob("*.gobr")}
    if not arquivos:
        raise ValueError(f"Nenhuma fonte GoBR em {pasta}")
    if not pasta_descricoes.is_dir():
        raise FileNotFoundError(pasta_descricoes)

    exemplos: list[Exemplo] = []
    descricoes = {p.stem: p for p in pasta_descricoes.glob("*.txt")}
    extras = sorted(set(descricoes) - set(arquivos))
    if extras:
        raise ValueError(f"Descrições sem fonte GoBR: {extras}")
    for identificador, caminho in sorted(descricoes.items()):
        explicacao = caminho.read_text(encoding="utf-8-sig").strip()
        if not explicacao:
            raise ValueError(f"Descrição vazia: {caminho}")
        fonte = arquivos[identificador]
        exemplos.append(Exemplo(
            id=identificador,
            explicacao=explicacao,
            codigo=fonte.read_text(encoding="utf-8-sig"),
            arquivo=fonte.relative_to(repo_dir).as_posix(),
        ))
    if not exemplos:
        raise ValueError("Nenhuma explicação válida encontrada")
    return exemplos, sorted(set(arquivos) - set(descricoes))


def diagnostico(repo_dir: Path) -> tuple[list[Exemplo], list[str]]:
    exemplos, ausentes = carregar_exemplos(repo_dir)
    print(f"Descrições comerciais: {len(exemplos)}")
    print(f"Fontes GoBR sem explicação: {len(ausentes)}")
    if ausentes:
        print("IDs ausentes:", ", ".join(ausentes))
    return exemplos, ausentes


def carregar_modelo(exemplos: list[Exemplo]):
    import torch
    from llama_index.embeddings.huggingface import HuggingFaceEmbedding
    from transformers import AutoTokenizer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    lote = 4 if device == "cuda" else 2
    precisao = "float16" if device == "cuda" else "float32"
    print("Dispositivo:", device, "| precisão:", precisao, "| lote:", lote)
    if device == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
    tokenizer = AutoTokenizer.from_pretrained(MODELO_EMBEDDING)
    maior_texto = max(len(tokenizer.encode(e.explicacao, truncation=False)) for e in exemplos)
    if maior_texto > LIMITE_TOKENS:
        raise ValueError(f"Descrição com {maior_texto} tokens excede o limite do BGE-M3 ({LIMITE_TOKENS})")
    comprimento_modelo = min(LIMITE_TOKENS, max(256, ((maior_texto + 255) // 128) * 128))
    print("Maior descrição:", maior_texto, "tokens | limite configurado:", comprimento_modelo)
    modelo = HuggingFaceEmbedding(
        model_name=MODELO_EMBEDDING,
        device=device,
        embed_batch_size=lote,
        max_length=comprimento_modelo,
        model_kwargs={"torch_dtype": "float16"} if device == "cuda" else {},
    )
    return tokenizer, modelo, {"precisao": precisao, "lote": lote, "max_length": comprimento_modelo}


def assinatura_indice(exemplos: list[Exemplo], revisao_modelo: str | None, config_execucao: dict) -> str:
    config = {
        "pipeline": VERSAO_PIPELINE,
        "embedding": MODELO_EMBEDDING,
        "revisao_modelo": revisao_modelo,
        "execucao": config_execucao,
        "limite": LIMITE_TOKENS,
        "versoes": {p: version(p) for p in (
            "llama-index-core", "llama-index-embeddings-huggingface",
            "sentence-transformers", "transformers",
        )},
        "dados": [(e.id, e.explicacao, e.codigo, e.arquivo)
                  for e in exemplos],
    }
    return hashlib.sha256(json.dumps(config, ensure_ascii=False, sort_keys=True).encode()).hexdigest()


def abrir_indice(repo_dir: Path, cache_dir: Path | None = None):
    """Cria ou reutiliza um índice de uma descrição completa por arquivo GoBR."""
    from llama_index.core import Settings, StorageContext, VectorStoreIndex, load_index_from_storage
    from llama_index.core.schema import MetadataMode, TextNode

    exemplos, ausentes = diagnostico(repo_dir)
    if ausentes:
        raise ValueError("Complete todas as descrições antes de criar o índice")
    tokenizer, modelo, config_execucao = carregar_modelo(exemplos)
    Settings.llm = None
    Settings.embed_model = modelo
    nos = []
    for exemplo in exemplos:
        metadados = {
            "fonte_id": exemplo.id,
            "arquivo": exemplo.arquivo,
        }
        no = TextNode(
            id_=exemplo.id,
            text=exemplo.explicacao,
            metadata=metadados,
            excluded_embed_metadata_keys=list(metadados),
            excluded_llm_metadata_keys=list(metadados),
        )
        tamanho = len(tokenizer.encode(
            no.get_content(metadata_mode=MetadataMode.EMBED), truncation=False,
        ))
        if tamanho > config_execucao["max_length"]:
            raise ValueError(f"Explicação {exemplo.id} tem {tamanho} tokens e excede o limite configurado; divida por regras antes de indexar")
        nos.append(no)

    cache = cache_dir or repo_dir.parent / "gobr_index_cache" / "explicacoes"
    revisao_modelo = getattr(tokenizer, "init_kwargs", {}).get("_commit_hash")
    pasta = cache / assinatura_indice(exemplos, revisao_modelo, config_execucao)
    manifesto = pasta / "manifesto.json"
    if manifesto.is_file():
        indice = load_index_from_storage(
            StorageContext.from_defaults(persist_dir=str(pasta)), embed_model=modelo,
        )
        print("Índice de explicações carregado:", pasta)
    else:
        pasta.mkdir(parents=True, exist_ok=True)
        indice = VectorStoreIndex(nos, embed_model=modelo, show_progress=True)
        indice.storage_context.persist(persist_dir=str(pasta))
        manifesto.write_text(json.dumps({
            "pipeline": VERSAO_PIPELINE,
            "modelo": MODELO_EMBEDDING,
            "execucao": config_execucao,
            "exemplos": len(exemplos),
            "ausentes": ausentes,
        }, ensure_ascii=False, indent=2), encoding="utf-8")
        print("Índice de explicações criado:", pasta)
    return indice, {e.id: e for e in exemplos}


def buscar(indice, por_id: dict[str, Exemplo], pedido: str, top_k: int = 5) -> list[dict]:
    if not pedido.strip() or top_k < 1:
        raise ValueError("Informe um pedido não vazio e top_k positivo")
    hits = indice.as_retriever(similarity_top_k=top_k).retrieve(pedido)
    retorno = []
    for hit in hits:
        exemplo = por_id[hit.node.metadata["fonte_id"]]
        retorno.append({
            "id": exemplo.id,
            "arquivo": exemplo.arquivo,
            "score": hit.score,
            "explicacao": exemplo.explicacao,
            "codigo_gobr": exemplo.codigo,
        })
    return retorno


CASOS_AVALIACAO = [
    ("Televendas: não agendar clientes inadimplentes e atender ativos a cada 7 dias", "regra_153"),
    ("Carteira interior com clientes ativos a cada 14 dias", "regra_136"),
    ("Clientes curva A com frequência de 3 dias para Yara", "regra_142"),
]


def avaliar(indice, por_id: dict[str, Exemplo], top_k: int = 5) -> list[dict]:
    """Casos iniciais: posição da referência conhecida, sem presumir exclusividade."""
    relatorio = []
    for pergunta, referencia in CASOS_AVALIACAO:
        hits = buscar(indice, por_id, pergunta, top_k)
        posicao = next((i for i, h in enumerate(hits, 1) if h["id"] == referencia), None)
        item = {"pergunta": pergunta, "referencia": referencia,
                "posicao_top_k": posicao, "retornos": [h["id"] for h in hits]}
        relatorio.append(item)
        print(json.dumps(item, ensure_ascii=False))
    return relatorio



## Cobertura e criação do índice

A primeira execução baixa o modelo BGE-M3 e gera embeddings. Com GPU T4, usa FP16 e lotes de 4 descrições; em CPU, usa FP32 e lotes de 2. As seguintes usam o cache quando dados e configuração não mudaram. O código GoBR não entra no texto vetorizado. Confira se o diagnóstico mostra 246 descrições e zero ausentes.

In [4]:
exemplos, ausentes = diagnostico(REPO_DIR)
indice, por_id = abrir_indice(REPO_DIR)

Descrições comerciais: 246
Fontes GoBR sem explicação: 0
Descrições comerciais: 246
Fontes GoBR sem explicação: 0
Dispositivo: cuda | precisão: float16 | lote: 4
GPU: Tesla T4


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

LLM is explicitly disabled. Using MockLLM.


Generating embeddings:   0%|          | 0/246 [00:00<?, ?it/s]

Índice de explicações criado: /content/gobr_index_cache/explicacoes/eaab24188b3727a1cb914fd9fc224566864d6e06cc9d25f02c271d0c4e5c9635


## Buscar por descrição comercial

Cada resultado traz a explicação usada na busca e o código original vinculado. Similaridade não comprova correção da regra.

In [5]:
pedido = 'Não agendar clientes inadimplentes de televendas e atender clientes ativos a cada 7 dias'
resultados = buscar(indice, por_id, pedido, top_k=5)
for item in resultados:
    print(item['id'], item['score'], item['arquivo'])
    print(item['explicacao'][:500])
    print(item['codigo_gobr'][:500])
    print()

regra_075 0.7977918196771658 dados/referencias/regra_075.gobr
Regra comercial de televendas

Considerar todos os vendedores de televendas e os clientes de suas carteiras fixas. Não agendar inadimplentes. Agendar clientes ativos e adimplentes a cada 7 dias, clientes inativos e adimplentes a cada 14 dias, e clientes sem venda a cada 14 dias. Aplicar nessa ordem.

version 1.0

time "time televendas":
    vendedores todostelevendas

carteira "carteira televendas":
    clientes carteirafixa("time televendas")

parametros "Regra Padrao da Empresa Customizavel: nao agendar inadimplentes":
    da carteira("carteira televendas")
    quando inadimplentes
    nao agendar

parametros "Regra Padrao da Empresa Customizavel: 7 dias para ativos adimplentes":
    da carteira("carteira televendas")
    quando ativos para o vendedor e adimplentes
    frequencia de 7 dia

regra_119 0.7939519703090334 dados/referencias/regra_119.gobr
Regra comercial da Ultrax

Considerar os vendedores de televendas habilit

## Avaliar recuperação

Três pedidos independentes mostram a posição de uma referência conhecida no top 5. Inspecione também os demais retornos; outras fontes podem ser relevantes.

In [6]:
relatorio = avaliar(indice, por_id, top_k=5)

{"pergunta": "Televendas: não agendar clientes inadimplentes e atender ativos a cada 7 dias", "referencia": "regra_153", "posicao_top_k": null, "retornos": ["regra_119", "regra_172", "regra_075", "regra_233", "regra_177"]}
{"pergunta": "Carteira interior com clientes ativos a cada 14 dias", "referencia": "regra_136", "posicao_top_k": null, "retornos": ["regra_181", "regra_220", "regra_038", "regra_055", "regra_008"]}
{"pergunta": "Clientes curva A com frequência de 3 dias para Yara", "referencia": "regra_142", "posicao_top_k": 1, "retornos": ["regra_142", "regra_141", "regra_161", "regra_211", "regra_190"]}
